# 06_compute_novelty_and_density

**Objective:** Compute patent-level novelty and competitive density from PaECTER embeddings by comparing each focal patent against the five-year prior-art window in the universe.

**Inputs:** `universe_emb_norm.f16.bin`; `universe_meta.parquet`; `patent_features_base.parquet`.

**Outputs:** `novelty_features.parquet` (`backward_sim`, `novelty_mean`, `novelty_min`, `novelty_top10`) and `novelty_raw.npz`.

In [ ]:
# Imports
from pathlib import Path
import time
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import torch
from tqdm import tqdm

## Paths, configuration and device selection

In [ ]:
# Paths, configuration, and device selection
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data").exists())
RAW  = ROOT / "data" / "raw"
PROC = ROOT / "data" / "processed"
PROC.mkdir(parents=True, exist_ok=True)
DATA_DIR = PROC
EMB_BIN  = PROC / "universe_emb_norm.f16.bin"
META     = PROC / "universe_meta.parquet"
FOCAL    = RAW / "embeddings" / "patent_features_base.parquet"
OUT      = PROC / "novelty_features.parquet"

DIM             = 1024
WINDOW_YEARS    = 5
TOP_K           = 10
CHUNK_FOCAL     = 256
EPOCH = pd.Timestamp("1970-01-01").toordinal()

DEVICE = "mps" if torch.backends.mps.is_available() else "cpu"
print(f"Device: {DEVICE}")

t0 = time.time()

## 1) Load universe metadata and memmap

In [ ]:
# Load universe metadata and memory-map embeddings
print("\n=== 1) Lade Universe-Metadata ===")
meta = pd.read_parquet(META)
N_universe = len(meta)
priority_days = meta["priority_date_days"].values
NULL_SENTINEL = np.iinfo(np.int32).min
n_dated = (priority_days != NULL_SENTINEL).sum()
print(f"  {N_universe:,} Universe-Patente, mit Datum: {n_dated:,}")

emb_mm = np.memmap(EMB_BIN, dtype=np.float16, mode="r", shape=(N_universe, DIM))

## 2) Load and normalise focal patents

In [ ]:
# Load and L2-normalise focal-patent embeddings
print("\n=== 2) Lade Focal-Patente ===")
focal = pd.read_parquet(FOCAL, columns=["lens_id","embedding","priority_date"])
focal["priority_date"] = pd.to_datetime(focal["priority_date"], errors="coerce")
n_total = len(focal)
focal_valid = focal[focal["priority_date"].notna()].reset_index(drop=True)
print(f"  total: {n_total:,}, mit priority_date: {len(focal_valid):,}")

focal_valid["priority_year"] = focal_valid["priority_date"].dt.year.astype("int32")

focal_emb_f32 = np.stack(focal_valid["embedding"].values).astype(np.float32)
norms = np.linalg.norm(focal_emb_f32, axis=1, keepdims=True)
norms[norms < 1e-8] = 1.0
focal_emb_f32 /= norms
focal_emb_f16 = focal_emb_f32.astype(np.float16)
print(f"  focal embedding shape: {focal_emb_f16.shape}, dtype={focal_emb_f16.dtype}")

focal_t = torch.from_numpy(focal_emb_f16).to(DEVICE)

## 3) Novelty computation per focal year

In [ ]:
# Compute novelty per focal year over the prior-art window
print("\n=== 3) Novelty-Computation pro Jahr ===")
years = sorted(focal_valid["priority_year"].unique())
print(f"  focal year range: {years[0]} - {years[-1]} ({len(years)} Jahre)")

results = np.full((len(focal_valid), 5), np.nan, dtype=np.float32)

t_compute = time.time()
for y in tqdm(years, desc="years"):
    win_start = pd.Timestamp(f"{y - WINDOW_YEARS}-01-01").toordinal() - EPOCH
    win_end   = pd.Timestamp(f"{y}-01-01").toordinal() - EPOCH

    mask = (priority_days >= win_start) & (priority_days < win_end) & (priority_days != NULL_SENTINEL)
    W_idx = np.where(mask)[0]
    M = len(W_idx)

    focal_year_mask = focal_valid["priority_year"].values == y
    focal_year_idx = np.where(focal_year_mask)[0]
    K = len(focal_year_idx)

    if M == 0 or K == 0:
        for fi in focal_year_idx:
            results[fi, 0] = 0
        continue

    W_np = np.ascontiguousarray(emb_mm[W_idx])
    W_t = torch.from_numpy(W_np).to(DEVICE)

    top_k = min(TOP_K, M)

    for c in range(0, K, CHUNK_FOCAL):
        idx_chunk = focal_year_idx[c:c+CHUNK_FOCAL]
        F_chunk = focal_t[idx_chunk]
        sims = F_chunk @ W_t.T

        mean_sim = sims.mean(dim=1)
        max_sim  = sims.max(dim=1).values
        topk_mean = sims.topk(top_k, dim=1).values.mean(dim=1)

        mean_np  = mean_sim.cpu().numpy()
        max_np   = max_sim.cpu().numpy()
        topk_np  = topk_mean.cpu().numpy()

        results[idx_chunk, 0] = M
        results[idx_chunk, 1] = mean_np
        results[idx_chunk, 2] = 1.0 - max_np
        results[idx_chunk, 3] = 1.0 - topk_np
        results[idx_chunk, 4] = 1.0 - mean_np

        del sims, mean_sim, max_sim, topk_mean

    del W_t
    if DEVICE == "mps":
        torch.mps.empty_cache()

print(f"  compute time: {(time.time()-t_compute)/60:.1f} min")

In [ ]:
# Save raw results to .npz as a fallback
np.savez(DATA_DIR / "novelty_raw.npz",
         results=results,
         lens_ids=focal_valid["lens_id"].values,
         priority_years=focal_valid["priority_year"].values,
         priority_dates=focal_valid["priority_date"].dt.strftime("%Y-%m-%d").values)
print(f"  raw saved -> novelty_raw.npz")

## 4) Write output parquet

In [ ]:
# Assemble and write the novelty feature table
print("\n=== 4) Schreibe Output ===")
out_df = pd.DataFrame({
    "lens_id":          focal_valid["lens_id"].values,
    "priority_date":    focal_valid["priority_date"].dt.date.values,
    "priority_year":    focal_valid["priority_year"].values,
    "n_priors":         pd.Series(results[:, 0]).astype("Int64"),
    "backward_sim":     results[:, 1],
    "novelty_mean":     results[:, 4],
    "novelty_min":      results[:, 2],
    "novelty_top10":    results[:, 3],
})
out_df.to_parquet(OUT, compression="zstd")
print(f"  -> {OUT} ({OUT.stat().st_size/1e6:.1f} MB)")

## 5) Quick statistics

In [ ]:
# Report summary statistics
print("\n=== Quick-Stats ===")
valid = out_df["backward_sim"].notna()
print(f"  focal mit Novelty (n_priors>=1): {valid.sum():,} / {len(out_df):,}")
print(f"\n  backward_sim:  mean={out_df.loc[valid,'backward_sim'].mean():.4f}, "
      f"median={out_df.loc[valid,'backward_sim'].median():.4f}, "
      f"std={out_df.loc[valid,'backward_sim'].std():.4f}")
print(f"  novelty_mean:  mean={out_df.loc[valid,'novelty_mean'].mean():.4f}, "
      f"p25={out_df.loc[valid,'novelty_mean'].quantile(0.25):.4f}, "
      f"p75={out_df.loc[valid,'novelty_mean'].quantile(0.75):.4f}")
print(f"  novelty_min:   mean={out_df.loc[valid,'novelty_min'].mean():.4f}, "
      f"p25={out_df.loc[valid,'novelty_min'].quantile(0.25):.4f}, "
      f"p75={out_df.loc[valid,'novelty_min'].quantile(0.75):.4f}")
print(f"  novelty_top10: mean={out_df.loc[valid,'novelty_top10'].mean():.4f}, "
      f"p25={out_df.loc[valid,'novelty_top10'].quantile(0.25):.4f}, "
      f"p75={out_df.loc[valid,'novelty_top10'].quantile(0.75):.4f}")
print(f"\n  total runtime: {(time.time()-t0)/60:.1f} min")